# Day 4 · Notebook 1 — Foundations: Setup, Invoke, and Read the Trace

**TravelMind** — an airline customer-support agent on Amazon Bedrock.
Use cases: **UC-A1** booking exception handler · **UC-A2** disruption replanning.

This is notebook 1 of 5. By the end you will:

- have a working Python environment and confirmed AWS access (from scratch)
- understand the **four** Bedrock clients and when to use each
- run the model **alone** with the Converse API (text in, text out)
- invoke the **managed** TravelMind agent you built on Day 3, from code
- read its **reasoning trace** (THINK / ACT / OBSERVE) in code

> The series: **1** foundations · **2** rebuild the loop by hand · **3** action groups (Lambda + return-control) · **4** controlling behavior (loops, inference, hallucinations) · **5** production practices.

Everything runs in **us-east-1** — that is the only region your training access is scoped to. A different region is the `AccessDenied` you fought in class.

## 0. Prerequisites — read this before running anything

Nothing here is assumed. You need all of the following once:

1. **An AWS account** with the Bedrock console open in **us-east-1** (N. Virginia).
2. **Model access granted.** Bedrock console → *Model access* → enable at least **Amazon Nova Lite** and **Anthropic Claude 3.5 Haiku**. Without this, every model call returns `AccessDeniedException`.
3. **The TravelMind agent from Day 3.** Bedrock console → *Agents* → open your agent → copy the **Agent ID** (10 characters) from *Agent overview*. The test alias is always `TSTALIASID`.
4. **IAM permissions** on whatever identity you run as. For this notebook the minimum is: `bedrock:ListFoundationModels`, `bedrock:InvokeModel`, `bedrock:Converse`, `bedrock-agent-runtime:InvokeAgent`, and `sts:GetCallerIdentity`. Your classroom user already has these.

If any of these is missing, the matching cell below will tell you exactly which one.

## How to run this notebook

**VS Code (local)** — 3 steps:
1. `python -m venv .venv` then activate it (`source .venv/bin/activate` on macOS/Linux, `.venv\Scripts\activate` on Windows).
2. Configure credentials once: `aws configure` (or set `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` / `AWS_SESSION_TOKEN` as environment variables). Set `AWS_DEFAULT_REGION=us-east-1`.
3. `pip install boto3` then select the `.venv` interpreter as the notebook kernel (top-right *Select Kernel*).

**Google Colab** — 3 steps:
1. Run the install cell below (`!pip install -q boto3`).
2. Put credentials in **Colab Secrets** (key icon in the left sidebar) or set them with `os.environ[...]` in a cell. **Never paste long-lived keys into a shared notebook.**
3. Keep `region_name="us-east-1"` everywhere (the config cell does this for you).

In [1]:
# If you are on Colab, uncomment the next line. On VS Code with an activated venv, skip it.
# !pip install -q boto3

import boto3, botocore, json, uuid, sys
print("python :", sys.version.split()[0])
print("boto3  :", boto3.__version__)
print("botocore:", botocore.__version__)
# boto3 >= 1.34 has Converse; >= 1.35 is comfortable for agents + guardrails. Upgrade if older:
#   pip install -U boto3

python : 3.14.5
boto3  : 1.43.21
botocore: 1.43.21


In [23]:
# ----------------------------------------------------------------------
# CONFIG — the only cell you edit. Everything downstream reads from here.
# ----------------------------------------------------------------------
REGION = "us-east-1"                 # the only region your access is scoped to

# From Bedrock console > Agents > (your TravelMind agent) > Agent overview:
AGENT_ID = "3KCNKSOI8U"              # <-- paste your 10-char Agent ID
ALIAS_ID = "TSTALIASID"             # 'TSTALIASID' is the always-present test alias

# Model ids in us-east-1.
NOVA_MODEL   = "amazon.nova-lite-v1:0"                          # works on-demand
CLAUDE_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"    # needs the 'us.' profile

# quick self-check so you fail fast with a clear message
_missing = []
if AGENT_ID == "XXXXXXXXXX":
    _missing.append("AGENT_ID is still the placeholder — paste your real Agent ID.")
print("CONFIG OK" if not _missing else "ACTION NEEDED:")
for m in _missing:
    print("  -", m)

CONFIG OK


### Credentials — how they are found

boto3 looks for credentials in this order: environment variables → shared config (`~/.aws/credentials` from `aws configure`) → an attached IAM role. You do **not** put keys in code.

**What changes in production:** there are no access keys at all. The workload runs under an **IAM role** — a Lambda execution role, an ECS task role, or an EC2 instance profile — and boto3 picks the role up automatically. Region comes from the environment, not a hard-coded string.

In [13]:
# Prove your credentials resolve, before doing anything Bedrock-specific.
sts = boto3.client("sts", region_name=REGION)
try:
    who = sts.get_caller_identity()
    ACCOUNT_ID = who["Account"]
    print("Account :", ACCOUNT_ID)
    print("Identity:", who["Arn"])
except botocore.exceptions.NoCredentialsError:
    print("No credentials found. VS Code: run `aws configure`. Colab: set env vars / Secrets.")
    ACCOUNT_ID = None

Account : 452203592848
Identity: arn:aws:iam::452203592848:user/IBSAgenticAIAdmin


## 1. The four Bedrock clients

This is the single most common 30-minute mistake. There are **four** clients on two planes:

| boto3 client | Plane | You use it for |
|---|---|---|
| `bedrock` | Control | list models, **create guardrails** |
| `bedrock-runtime` | Inference | `converse`, `invoke_model`, **`apply_guardrail`** |
| `bedrock-agent` | Control (agents) | create / update / **prepare** agent, **action groups** |
| `bedrock-agent-runtime` | Data (agents) | **`invoke_agent`**, `retrieve`, `retrieve_and_generate` |

Rule of thumb: **build it** = `bedrock-agent`; **talk to it** = `bedrock-agent-runtime`.

In [14]:
bedrock        = boto3.client("bedrock",               region_name=REGION)  # control: models, guardrails
bedrock_runtime= boto3.client("bedrock-runtime",       region_name=REGION)  # inference: converse, apply_guardrail
agent_build    = boto3.client("bedrock-agent",         region_name=REGION)  # control: agents, action groups
agent_runtime  = boto3.client("bedrock-agent-runtime", region_name=REGION)  # data: invoke_agent

print("clients ready:")
for c in (bedrock, bedrock_runtime, agent_build, agent_runtime):
    print("  -", c.meta.service_model.service_name)

clients ready:
  - bedrock
  - bedrock-runtime
  - bedrock-agent
  - bedrock-agent-runtime


## 2. Sanity check — list foundation models

A control-plane call that proves your `bedrock` client and credentials work, and shows the exact model ids available to you. If this 403s, your model access (prereq #2) or IAM is the problem.

In [ ]:
WATCH = {
    "amazon.nova-lite-v1:0",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0"
}

models = bedrock.list_foundation_models()["modelSummaries"]
found  = {m["modelId"] for m in models}

rows = [(
    m.get("modelLifecycle", {}).get("status", "UNKNOWN"),
    m["providerName"],
    m["modelId"],
    ",".join(m.get("inferenceTypesSupported", [])) or "-",
) for m in models]

orders = {"LEGACY": 0, "UNKNOWN": 1, "ACTIVE": 2}
rows.sort(key=lambda r: (orders.get(r[0], 1), r[1], r[2]))

print(f"{'STATUS':<8} {'PROVIDER':<11} {'MODEL ID':<48} INFERENCE")
print("-" * 95)
for status, provider, mid, inf in rows:
    if status == "ACTIVE" and mid not in WATCH:
        continue                                            # hide the healthy noise
    flag = " <-- WATCHED" if mid in WATCH else ""
    if status != "ACTIVE":
        flag = " <-- NOT ACTIVE" + (" + WATCHED" if mid in WATCH else "")
    print(f"{status:<8} {provider:<11} {mid:<48} {inf}{flag}")

for mid in WATCH:
    if mid not in found:
        print(f"MISSING: {mid} not offered in us-east-1 (EOL, wrong region, or not enabled).")

# ids = [m["modelId"] for m in models["modelSummaries"]]
# print("total models visible:", len(ids))
# print("\na few Nova + Claude ids:")
# for mid in ids:
#     if "nova" in mid or "claude" in mid:
#         print("  ", mid)
# Note: bare ids here (e.g. anthropic.claude-3-5-haiku-...) are NOT always callable on-demand.
# Claude in us-east-1 must be called through the 'us.' inference profile (see section 3b).

STATUS   PROVIDER    MODEL ID                                         INFERENCE
-----------------------------------------------------------------------------------------------
LEGACY   AI21 Labs   ai21.jamba-1-5-large-v1:0                        ON_DEMAND <-- NOT ACTIVE
LEGACY   AI21 Labs   ai21.jamba-1-5-mini-v1:0                         ON_DEMAND <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-canvas-v1:0                          ON_DEMAND,PROVISIONED <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-premier-v1:0                         INFERENCE_PROFILE <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-premier-v1:0:1000k                   - <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-premier-v1:0:20k                     - <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-premier-v1:0:8k                      - <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-premier-v1:0:mm                      - <-- NOT ACTIVE
LEGACY   Amazon      amazon.nova-reel-v1:0                            ON_DEM

KeyboardInterrupt: 

## 3. The model alone — text in, text out

Before agents, understand the thing agents are built around. A foundation model is a **stateless function**: you send messages, you get text back. No memory, no tools, no loop. That machinery is Bedrock's job, not the model's.

`Converse` is the unified inference API. One shape works across model families.

In [ ]:
def converse_text(model_id, prompt, system=None, temperature=0.0, max_tokens=512):
    """Single-shot Converse call. Returns the model's text."""
    kwargs = {
        "modelId": model_id,
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
    }
    if system:
        kwargs["system"] = [{"text": system}]
    resp = bedrock_runtime.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"]

# LLM Call
# Nova Lite works on-demand in us-east-1 — good for a first call.
print(converse_text(NOVA_MODEL, "In one sentence, what does an airline PNR identify?"))

An airline PNR (Passenger Name Record) identifies and stores all the details and information related to a passenger's flight reservation, including personal information, itinerary, ticketing, and other relevant data.


### 3b. The `us.` prefix trap (the one that cost class time)

Some model families — Claude here — are **only reachable through a cross-region inference profile** in this region. The profile id starts with `us.`. Call the bare id and you get `ValidationException` or `AccessDenied`. The cell below shows both so you remember the failure shape.

In [32]:
BARE_CLAUDE = "anthropic.claude-haiku-4-5-20251001-v1:0"   # no 'us.' -> expect failure

# 1) bare id (fails, but now with the *intended* lesson: "use an inference profile")
try:
    print("bare id:", converse_text(BARE_CLAUDE, "Say OK."))
except botocore.exceptions.ClientError as e:
    print("bare id FAILED ->", e.response["Error"]["Code"])
    print("   message:", e.response["Error"]["Message"][:160])

# 2) the 'us.' inference profile (works)
print("\nprofile id:", converse_text(CLAUDE_MODEL, "Say OK in three words."))

bare id FAILED -> ValidationException
   message: Invocation of model ID anthropic.claude-haiku-4-5-20251001-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference

profile id: Okay, I understand.


## 4. Useful use case — talk to the *managed* TravelMind agent

Now the agent you built on Day 3. UC-A2: a passenger's flight was disrupted and wants options. The agent decides which tools to call, calls them, and writes the answer — all managed by Bedrock.

`invoke_agent` lives on `bedrock-agent-runtime`. Its response key is **`completion`**, an event **stream** you iterate — it is not a dict you index into.

In [33]:
def ask_agent(prompt, session_id=None, trace=False):
    """Invoke the managed agent. Returns (answer, traces, session_id)."""
    session_id = session_id or str(uuid.uuid4())
    resp = agent_runtime.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=ALIAS_ID,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=trace,
    )
    answer, traces = "", []
    for event in resp["completion"]:          # iterate the STREAM
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode("utf-8")
        elif "trace" in event:
            traces.append(event["trace"]["trace"])
    return answer, traces, session_id

answer, _, sid = ask_agent(
    "My flight on PNR ABC123 was disrupted. What are my options?"
)
print("SESSION:", sid)
print(answer)

SESSION: 163cbba7-e27b-4784-8555-7cd9e6e1dce5
Your flight was cancelled. Here are your rebooking options: AI-318 at 18:40 or 6E-552 at 21:15.


## 5. Read the trace — THINK / ACT / OBSERVE in code

`enableTrace=True` delivers the same *Show trace* panel you read in the console, as data. The orchestration trace has three pieces you care about:

- `rationale` — the model **thinking**
- `invocationInput` — the model **acting** (which tool, with what input)
- `observation` — the result coming back / the final answer

In [26]:
answer, traces, _ = ask_agent(
    "My flight on PNR ABC123 was disrupted. What are my options?",
    trace=True,
)

for t in traces:
    ot = t.get("orchestrationTrace")
    if not ot:
        continue
    if "rationale" in ot:
        print("THINK:", ot["rationale"]["text"][:300])
    if "invocationInput" in ot:
        print("ACT  :", json.dumps(ot["invocationInput"])[:300])
    if "observation" in ot:
        obs = ot["observation"]
        if obs.get("finalResponse"):
            print("FINAL:", obs["finalResponse"]["text"][:400])
    print("-" * 60)

------------------------------------------------------------
------------------------------------------------------------
THINK: The User's goal is to know the rebooking options for their disrupted flight.
(2) The User has provided the PNR code.
(3) The best action plan is to first look up the booking, then find the disruption reason, and finally get rebooking options.
(4) The first step is to look up the booking.
(5) The ava
------------------------------------------------------------
ACT  : {"actionGroupInvocationInput": {"actionGroupName": "BookingActions", "executionType": "LAMBDA", "function": "lookup_booking", "parameters": [{"name": "pnr", "type": "string", "value": "ABC123"}]}, "invocationType": "ACTION_GROUP", "traceId": "9811a344-5f11-4da2-adc7-c8a26bb9d8c4-0"}
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
-----------------------------------

## 6. Session memory — the same `sessionId` continues the conversation

Reuse the session id and the agent remembers the prior turns. Use a fresh id to start clean. This is the memory handle you do *not* get for free in the hand-built loop (notebook 2 — you manage it yourself).

In [27]:
ans1, _, sid = ask_agent("My PNR is ABC123. Was my flight disrupted?")
print("Turn 1:", ans1[:300], "\n")

# same sid -> 'it' resolves to PNR ABC123 from turn 1
ans2, _, _ = ask_agent("What are my rebooking options then?", session_id=sid)
print("Turn 2:", ans2[:400])

Turn 1: Your flight was disrupted due to heavy fog at the origin. Would you like to rebook? 

Turn 2: Your rebooking options are: Flight AI-318 departing at 18:40 or Flight 6E-552 departing at 21:15.


## Recap

- **Four clients, two planes.** Talk to the agent through `bedrock-agent-runtime`.
- The model is a **stateless text function**; Bedrock is the runtime around it.
- `invoke_agent` → `completion` is a **stream**; `enableTrace=True` gives you THINK / ACT / OBSERVE.
- Claude in us-east-1 needs the **`us.`** inference profile.

**What changes in production:** no access keys (IAM role), region from the environment, a numbered alias instead of `TSTALIASID`, and every call wrapped in retry + logging (notebook 5).

**Next — Notebook 2:** rebuild this agent's loop by hand with Converse + tools, so you can see and control every step.